In [ ]:
import getpass
from pathlib import Path
from Physical_Quantity.data_records.Physical_Data_Record import (
    DensityRecord, CorrelationRecord, SuperCorrRecord)
from Physical_Quantity.physical_analysis.Density_Process import DensityProcessor
from Physical_Quantity.physical_analysis.Correlation_Process import CorrelationProcessor
from Physical_Quantity.physical_analysis.Super_Correlation_Process import SuperCorrProcessor

## Itensor计算密度平均值<n>的数据提取与作图

In [ ]:
### 基本参数
N = 200  # 格点数
a = 1.0  # honeycomb晶格常数
data_n = getpass.getpass("请输入密度数据<n>: ")   # getpass默认不显示输入字符，适用于长数据粘贴

### 将输入的数据封装成DensityRecord对象列表
n_Itensor = DensityRecord.extract_data(data_n, N, a)

## 打印n的前5个数据
print("\n----------------------------- honeycomb晶格点的电子密度分布数据 -----------------------------------")
for record in n_Itensor[:5]:
    print(record)
print(f"... 还有{len(n_Itensor)-5}个点")

### 电子密度的一维折线图可视化
DensityProcessor.Line_Chart_Visualization(n_Itensor)

## UltraDMRG相对于Itensor的密度绝对误差

In [ ]:
### ================= 文件夹路径变量（需要修改的部分）=====================
N = 200                  # 格点数
a = 1.0                  # honeycomb晶格常数
δ = "d1_10"              # 空穴率
bond_dim = "1500"        # bond dimension
t_J = "t_J_3"            # t/J
version = "49"           # 子晶格文件夹

### UltraDMRG 数据（默认取 complex 计算结果的 1500 档）
base_dir = Path(f'Data/complex/Data_N{N}/{t_J}/{δ}/{bond_dim}_measure_phys_qty/measure_{version}')
data_n = (base_dir / "nf.json").read_text(encoding='utf-8')
n_UltraDMRG = DensityRecord.extract_data(data_n, N, a)

### UltraDMRG相对于Itensor的密度绝对误差可视化
directions = {
    'UltraDMRG': n_UltraDMRG,
    'Itensor': n_Itensor
}
DensityProcessor.Line_Chart_Visualization_difference(directions)

## Itensor计算A、B子格自旋关联<S_i·S_j>的数据提取与作图

In [ ]:
### 基本参数
N = 200  # 格点数
a = 1.0  # honeycomb晶格常数

### 配置两个子格：名称、源点偏移、最大场点偏移
configs = [
    ("A", 1, -3),   # A子格：源点偏移+1, 最大场点偏移-3
    ("B", 2, -2),   # B子格：源点偏移+2, 最大场点偏移-2
]

### 保存A、B子格的关联函数对象
results = {}

### 将输入的数据提取并封装成CorrelationRecord对象列表
for name, src_off, max_off in configs:
    ## 输入数据
    data_SiSj = getpass.getpass(f"请输入{name}子格<S_i·S_j>: ")

    ## 生成子格序列
    source_point = int(N / 8 / 4) * 8 + src_off
    field_point_max = int(N / 4 * 3) + max_off
    sequence = range(source_point, field_point_max + 1, 4)

    ## 数据封装与打印
    SiSj = CorrelationRecord.extract_data(data_SiSj, sequence, N, a)
    print(f"\n----------------------------- {name}子格 自旋关联函数数据 -----------------------------------")
    for record in SiSj[:5]:
        print(record)
    print(f"... 还有{len(SiSj)-5}个点")

    results[name] = SiSj

### 两个子格统一作图（2 行 × 3 列）
CorrelationProcessor.Spin_Corr_Visualization(
    {f'{name} sublattice': recs for name, recs in results.items()}
)

### 如需分别引用：
SiSj_A_Itensor = results["A"]
SiSj_B_Itensor = results["B"]

## UltraDMRG相对于Itensor的自旋关联绝对误差

In [ ]:
### ================= 文件夹路径变量（需要修改的部分）=====================
N = 200                  # 格点数
a = 1.0                  # honeycomb晶格常数
δ = "d1_10"              # 空穴率
bond_dim = "1500"        # bond dimension
t_J = "t_J_3"            # t/J
version_A = "49"         # A子晶格文件夹
version_B = "50"         # B子晶格文件夹

### 配置两个子格：名称、版本号、源点偏移、最大场点偏移
configs = [
    ("A", version_A, 1, -3),   # A子格：version=49, 源点偏移+1, 最大场点偏移-3
    ("B", version_B, 2, -2),   # B子格：version=50, 源点偏移+2, 最大场点偏移-2
]

### UltraDMRG 数据（默认取 complex 计算结果的 1500 档）
results = {}
for name, version, src_off, max_off in configs:
    base_dir = Path(f'Data/complex/Data_N{N}/{t_J}/{δ}/{bond_dim}_measure_phys_qty/measure_{version}')
    data_SuSd = (base_dir / f'sm{version}sp.json').read_text(encoding='utf-8')
    data_SdSu = (base_dir / f'sp{version}sm.json').read_text(encoding='utf-8')
    data_SzSz = (base_dir / f'sz{version}sz.json').read_text(encoding='utf-8')

    source_point = int(N / 8 / 4) * 8 + src_off
    field_point_max = int(N / 4 * 3) + max_off
    sequence = range(source_point, field_point_max + 1, 4)

    SzSz = CorrelationRecord.extract_data(data_SzSz, sequence, N, a)
    SuSd = CorrelationRecord.extract_data(data_SdSu, sequence, N, a)  # 注意交叉对应
    SdSu = CorrelationRecord.extract_data(data_SuSd, sequence, N, a)
    results[name] = CorrelationProcessor.Combined_spin_correlation(SzSz, SuSd, SdSu)

SiSj_A_Ultra = results["A"]
SiSj_B_Ultra = results["B"]

### UltraDMRG相对于Itensor的自旋关联绝对误差可视化
directions = {
    'UltraDMRG': {'A sublattice': SiSj_A_Ultra,
                  'B sublattice': SiSj_B_Ultra},
    'Itensor': {'A sublattice': SiSj_A_Itensor,
                'B sublattice': SiSj_B_Itensor},
}
CorrelationProcessor.Spin_Corr_Visualization_difference(directions)

## Itensor计算超导关联<Δ_ij^†·Δ_kl>三键方向的数据提取与作图

In [ ]:
### 基本参数
N = 200  # 格点数
a = 1.0  # honeycomb晶格常数

### 三个方向与可视化的label的映射
directions = {
    'A': "target A",
    'B': "target B",
    'C': "target C"
}

### 输入数据并封装成SuperCorrRecord对象
super_corr_dict_Itensor = {}
for bond, label in directions.items():
    data = getpass.getpass(f"请输入文件scs_fix_{bond}的数据: ")
    super_corr_dict_Itensor[label] = SuperCorrRecord.extract_data(data, N, a)

### 数据可视化
SuperCorrProcessor.Super_Corr_Visualization_three(super_corr_dict_Itensor)

## UltraDMRG相对于Itensor的超导关联绝对误差

In [ ]:
### ================= 文件夹路径变量（需要修改的部分）=====================
N = 200                  # 格点数
a = 1.0                  # honeycomb晶格常数
δ = "d1_10"              # 空穴率
version = "50_54"        # reference bond
bond_dim = "1500"        # bond dimension
t_J = "t_J_3"            # t/J

### UltraDMRG 数据（默认取 complex 计算结果的 1500 档）
base_dir = Path(f'Data/complex/Data_N{N}/{t_J}/{δ}/{bond_dim}_measure_super_corr/super_corr_{version}')

### 将三个键方向A、B、C的数据提取并封装成SuperCorrRecord对象列表
records = {}
for bond in 'ABC':
    for channel in 'abcd':
        key = f'scs{bond}{channel}'
        fname = f'{key}_fix_refbond_{version}.json'
        records[key] = SuperCorrRecord.extract_data(
            (base_dir / fname).read_text(encoding='utf-8'), N, a
        )

### 方向显示名映射字典
bond_labels = {
    'A': "target A",
    'B': "target B",
    'C': "target C",
}

### 检测超导关联对称性并组合超导关联函数
super_corr_dict_UltraDMRG = {}
for bond, label in bond_labels.items():
    recs = [records[f'scs{bond}{channel}'] for channel in 'abcd']

    ## 对称性检查
    err_ad, err_bc = SuperCorrProcessor.symmetry_check(*recs)
    print(f"{bond}键方向的通道a与通道d的相对误差为：{err_ad}；通道b与通道c的相对误差为：{err_bc}")

    ## 组合超导关联函数，存入最终字典
    super_corr_dict_UltraDMRG[label] = SuperCorrProcessor.Combined_super_correlation(*recs)

### UltraDMRG相对于Itensor的超导关联函数的绝对误差可视化
directions = {
    'UltraDMRG': super_corr_dict_UltraDMRG,
    'Itensor': super_corr_dict_Itensor
}
SuperCorrProcessor.Super_Corr_Visualization_difference(directions)